# FloodRisk Sri Lanka - MERIT Hydro × HydroBASINS compatibility and seven Kelani upstream watersheds

**Project:** J26-DS-314  
**Scope:** National visual compatibility screen for 18 named research basins, then **seven Kelani Ganga gauges only**.  
**Outputs per gauge:** (1) irregular upstream catchment GeoJSON, (2) native-grid catchment mask GeoTIFF, and (3) bounding-box GeoJSON.

## Our workflow and evidence policy

1. We overlay the nationwide MERIT Hydro river network and WWF HydroBASINS Level 12 sub-basin boundaries on the two basemaps **OpenStreetMap and Esri Satellite**. We show one representative ArcGIS-dashboard station for each of the 18 planned research basins. These markers do **not** imply that Level 12 polygons have those basin names.
2. We use the seven **ArcGIS GeoJSON geometry points** supplied by the Irrigation Department's public dashboard. Original coordinates never change. A second, separately recorded *computational outlet* is selected on MERIT's flow grid.
3. We download only the required **Kelani regional** MERIT `dir` and `upa` pixels, on the **exact original grid**. The nationwide layers are map tiles and **are not downloaded**. MERIT supplies ESRI-style D8 flow directions.
4. We trace upstream connectivity from each selected outlet using a free, transparent Python implementation of the same D8 rule used by watershed tools (no paid software). We do not re-derive flow direction from a DEM.
5. We calculate catchment polygons, masks and bounding rectangles, audit area, outlet distance, border truncation and a HydroBASINS cross-check. Each derived polygon remains **CANDIDATE**, even if automatic checks pass, manual river/confluence inspection is required before claiming verified boundaries.

**Key methodological clarification:** MERIT and HydroBASINS describe similar hydrography but are **different products**. Level 12 subbasins are not the exact watersheds of the gauge points. This is a consistency review, not a numerical equality test.

Official references: [MERIT Hydro v1.0.1](https://developers.google.com/earth-engine/datasets/catalog/MERIT_Hydro_v1_0_1), [HydroBASINS level 12](https://developers.google.com/earth-engine/datasets/catalog/WWF_HydroSHEDS_v1_Basins_hybas_12), [WhiteboxTools D8 watershed description](https://jblindsay.github.io/wbt_book/available_tools/hydrological_analysis.html).

## A. Environment, sources and gauge registry

In [1]:
import json, math, time, warnings
from collections import deque
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import folium
from folium import plugins
import requests
import rasterio
from rasterio.windows import Window, transform as window_transform
from rasterio.features import shapes
from rasterio.transform import rowcol, xy, Affine
from shapely.geometry import shape, mapping, box, Point
from shapely.ops import unary_union
from pyproj import Geod
import ee
from IPython.display import display

REPO_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
OUTPUT_DIR = REPO_ROOT / 'results' / 'catchments' / 'kelani_merit_hybas12_pilot'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Outputs:', OUTPUT_DIR.resolve())

EE_PROJECT = 'multimodal-flood-risk'
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('Earth Engine initialized:', EE_PROJECT)

Outputs: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\kelani_merit_hybas12_pilot
Earth Engine initialized: multimodal-flood-risk


In [2]:
# ArcGIS FeatureCollection geometry, source longitude/latitude (not rounded properties).
KELANI_GAUGES = pd.DataFrame(json.loads(r'''[{"station":"Nagalagam Street","tributary":"Kelani Ganga","lon":79.8786420000162,"lat":6.95826500005286,"official_area_km2":2085},{"station":"Hanwella","tributary":"Kelani Ganga","lon":80.0831260004171,"lat":6.90970000025326,"official_area_km2":1782},{"station":"Glencourse","tributary":"Kelani Ganga","lon":80.1942470000898,"lat":6.97698100015286,"official_area_km2":1463},{"station":"Kithulgala","tributary":"Kelani Ganga","lon":80.419213000397,"lat":6.99125899991869,"official_area_km2":383},{"station":"Holombuwa","tributary":"Gurugoda Oya","lon":80.2663310003202,"lat":7.18820100028799,"official_area_km2":155},{"station":"Deraniyagala","tributary":"Seethawaka Ganga","lon":80.3392120002439,"lat":6.92593399984068,"official_area_km2":183},{"station":"Norwood","tributary":"Kehelgamu Oya","lon":80.6113009999703,"lat":6.84014500042306,"official_area_km2":97}]'''))
RESEARCH_BASIN_MARKERS = pd.DataFrame(json.loads(r'''[{"basin":"Kelani Ganga","station":"Nagalagam Street","lon":79.8786420000162,"lat":6.95826500005286},{"basin":"Kalu Ganga","station":"Putupaula","lon":80.0615110003587,"lat":6.6143240002055},{"basin":"Gin Ganga","station":"Baddegama","lon":80.1753810002093,"lat":6.17380599991107},{"basin":"Nilwala Ganga","station":"Panadugama","lon":80.4823839999218,"lat":6.11178000039592},{"basin":"Walawe Ganga","station":"Moraketiya","lon":80.901243999799,"lat":6.34533899959541},{"basin":"Kirindi Oya","station":"Thanamalwila","lon":81.1337510002852,"lat":6.4690499999262},{"basin":"Menik Ganga","station":"Katharagama","lon":81.3321899999841,"lat":6.417705000347},{"basin":"Kumbukkan Oya","station":"Nakkala","lon":81.2970650001108,"lat":6.8953620001963},{"basin":"Heda Oya","station":"Siyambalanduwa","lon":81.545409999742,"lat":6.90591099956999},{"basin":"Maduru Oya","station":"Padiyathalawa","lon":81.1946370001375,"lat":7.38225100016999},{"basin":"Mahaweli Ganga","station":"Manampitiya","lon":81.0905939999633,"lat":7.91298699959734},{"basin":"Yan Oya","station":"Horowpothana","lon":80.8808149995961,"lat":8.57699799998582},{"basin":"Ma Oya","station":"Yaka Wewa","lon":80.6749679997877,"lat":8.72603100017587},{"basin":"Malwathu Oya","station":"Thanthirimale","lon":80.2780700002985,"lat":8.59008099995541},{"basin":"Mee Oya","station":"Galgamuwa","lon":80.2623219997686,"lat":7.96936799958593},{"basin":"Deduru Oya","station":"Moragaswewa","lon":79.99659,"lat":7.70003500000002},{"basin":"Maha Oya","station":"Badalgama","lon":79.9839160002518,"lat":7.30074100013236},{"basin":"Aththanagalu Oya","station":"Dunamale","lon":80.0840849995936,"lat":7.11641600021263}]'''))
assert len(KELANI_GAUGES) == 7 and KELANI_GAUGES.station.nunique() == 7
assert len(RESEARCH_BASIN_MARKERS) == 18
assert KELANI_GAUGES.lon.between(79,82).all() and KELANI_GAUGES.lat.between(6,9).all()
assert all(KELANI_GAUGES.official_area_km2 > 0)
print('7 Kelani stations; 18 basin-level representative points:')
display(KELANI_GAUGES)

# All 40 original ArcGIS dashboard geometry points (WGS84 lon/lat).
# This registry is for the nationwide MAP only. Keep the 18-basin registry intact
# because Section C's basin-compatibility screen intentionally uses 18 sites.
ALL_ARCGIS_GAUGES = pd.DataFrame([
    ('Nagalagam Street', 'Kelani Ganga', 79.8786420000162, 6.95826500005286),
    ('Hanwella', 'Kelani Ganga', 80.0831260004171, 6.90970000025326),
    ('Glencourse', 'Kelani Ganga', 80.1942470000898, 6.97698100015286),
    ('Kithulgala', 'Kelani Ganga', 80.419213000397, 6.99125899991869),
    ('Holombuwa', 'Kelani Ganga', 80.2663310003202, 7.18820100028799),
    ('Deraniyagala', 'Kelani Ganga', 80.3392120002439, 6.92593399984068),
    ('Norwood', 'Kelani Ganga', 80.6113009999703, 6.84014500042306),
    ('Putupaula', 'Kalu Ganga', 80.0615110003587, 6.6143240002055),
    ('Ellagawa', 'Kalu Ganga', 80.2182539999508, 6.7315540003244),
    ('Rathnapura', 'Kalu Ganga', 80.3972350001154, 6.67906700028711),
    ('Kalawellawa (Millakanda)', 'Kalu Ganga', 80.1606899996635, 6.63116599994976),
    ('Magura', 'Kalu Ganga', 80.2439349995639, 6.51372700024646),
    ('Baddegama', 'Gin Ganga', 80.1753810002093, 6.17380599991107),
    ('Thawalama', 'Gin Ganga', 80.3306159997305, 6.34177600033244),
    ('Panadugama', 'Nilwala Ganga', 80.4823839999218, 6.11178000039592),
    ('Pitabeddara', 'Nilwala Ganga', 80.4766040004319, 6.21374599983565),
    ('Urawa', 'Nilwala Ganga', 80.5720109999098, 6.23672999982568),
    ('Thalgahagoda', 'Nilwala Ganga', 80.5329300000867, 6.01029999964291),
    ('Moraketiya', 'Walawe Ganga', 80.901243999799, 6.34533899959541),
    ('Thanamalwila', 'Kirindi Oya', 81.1337510002852, 6.4690499999262),
    ('Wellawaya', 'Kirindi Oya', 81.1121400004247, 6.69575999959604),
    ('Kuda Oya', 'Kirindi Oya', 81.1233370000797, 6.52474599995434),
    ('Katharagama', 'Menik Ganga', 81.3321899999841, 6.417705000347),
    ('Nakkala', 'Kumbukkan Oya', 81.2970650001108, 6.8953620001963),
    ('Siyambalanduwa', 'Heda Oya', 81.545409999742, 6.90591099956999),
    ('Ampara', 'Gal Oya', 81.6703430002516, 7.2907110004084),
    ('Padiyathalawa', 'Maduru Oya', 81.1946370001375, 7.38225100016999),
    ('Manampitiya', 'Mahaweli Ganga', 81.0905939999633, 7.91298699959734),
    ('Weraganthota', 'Mahaweli Ganga', 80.9891099999962, 7.31646999997607),
    ('Peradeniya', 'Mahaweli Ganga', 80.5930050000022, 7.26905900032262),
    ('Nawalapitiya', 'Mahaweli Ganga', 80.53616208997, 7.04894547008318),
    ('Thaldena', 'Mahaweli Ganga', 81.048886000161, 7.09065299961578),
    ('Horowpothana', 'Yan Oya', 80.8808149995961, 8.57699799998582),
    ('Yaka Wewa', 'Ma Oya', 80.6749679997877, 8.72603100017587),
    ('Thanthirimale', 'Malwathu Oya', 80.2780700002985, 8.59008099995541),
    ('Galgamuwa', 'Mee Oya', 80.2623219997686, 7.96936799958593),
    ('Moragaswewa', 'Deduru Oya', 79.99659, 7.70003500000002),
    ('Badalgama', 'Maha Oya', 79.9839160002518, 7.30074100013236),
    ('Giriulla', 'Maha Oya', 80.1250950003353, 7.33073600009912),
    ('Dunamale', 'Aththanagalu Oya', 80.0840849995936, 7.11641600021263),
 ], columns=['station', 'basin', 'lon', 'lat'])
assert len(ALL_ARCGIS_GAUGES) == 40 and ALL_ARCGIS_GAUGES.station.nunique() == 40
print('Nationwide map markers:',len(ALL_ARCGIS_GAUGES),'gauge points /',ALL_ARCGIS_GAUGES.basin.nunique(),'dashboard basin groups')


7 Kelani stations; 18 basin-level representative points:


,station,tributary,lon,lat,official_area_km2
0,Nagalagam Street,Kelani Ganga,79.878642,6.958265,2085
1,Hanwella,Kelani Ganga,80.083126,6.909700,1782
2,Glencourse,Kelani Ganga,80.194247,6.976981,1463
3,Kithulgala,Kelani Ganga,80.419213,6.991259,383
4,Holombuwa,Gurugoda Oya,80.266331,7.188201,155
5,Deraniyagala,Seethawaka Ganga,80.339212,6.925934,183
6,Norwood,Kehelgamu Oya,80.611301,6.840145,97


Nationwide map markers: 40 gauge points / 19 dashboard basin groups


## B. Nationwide compatibility overview (two basemaps, separate map)

We display **MERIT rivers** derived from `upa` (upstream drainage area) and **all Level 12 HydroBASINS sub-basin outlines intersecting Sri Lanka**. We use OpenStreetMap and Esri Satellite as togglable basemaps. The river network is a visualization threshold, not a new hydrology dataset. The 18 markers denote our named study basins via a representative gauge in each.

**How we review the map:** Toggle MERIT, HydroBASINS and marker layers independently. Zoom into Kelani, Kalu, Gin, Nilwala, Mahaweli and smaller basins, checking stream alignment, drainage divides, and any obvious offsets. A map overlay alone cannot prove watershed compatibility or coordinate correctness at 90 m resolution.

In [3]:
# Datasets and factual metadata checks.
MERIT = ee.Image('MERIT/Hydro/v1_0_1')
HYB = ee.FeatureCollection('WWF/HydroSHEDS/v1/Basins/hybas_12')
SRI_LANKA = (ee.FeatureCollection('FAO/GAUL/2015/level0')
             .filter(ee.Filter.eq('ADM0_NAME','Sri Lanka')))
assert int(SRI_LANKA.size().getInfo()) == 1, 'Country polygon not resolved; do not create a misleading national map.'
merit_bandnames = MERIT.bandNames().getInfo()
assert all(x in merit_bandnames for x in ['dir','upa','viswth'])
print('MERIT bands:',merit_bandnames)
print('MERIT original projection:',MERIT.select('dir').projection().getInfo())
print('HydroBASINS Level 12 intersecting Sri Lanka:',HYB.filterBounds(SRI_LANKA.geometry()).size().getInfo())
print('Example HydroBASINS properties:',
      list(ee.Feature(HYB.filterBounds(SRI_LANKA.geometry()).first()).propertyNames().getInfo()))

MERIT bands: ['elv', 'dir', 'wth', 'wat', 'upa', 'upg', 'hnd', 'viswth']
MERIT original projection: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.0008333333333333334, 0, -180.00041666666667, 0, -0.0008333333333333334, 84.99958333333333]}
HydroBASINS Level 12 intersecting Sri Lanka: 500
Example HydroBASINS properties: ['SUB_AREA', 'COAST', 'PFAF_ID', 'DIST_MAIN', 'HYBAS_ID', 'DIST_SINK', 'NEXT_DOWN', 'ORDER', 'ENDO', 'MAIN_BAS', 'NEXT_SINK', 'SORT', 'system:index', 'UP_AREA']


In [ ]:
# %pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


### Basemaps and CARTO configuration

In [11]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENABLE_CARTO_VOYAGER = True

ENV_FILE = Path("secrets.env").resolve()
loaded = load_dotenv(ENV_FILE)
CARTO_API_KEY = os.getenv("FLOODRISK_CARTO_BASEMAP_KEY", "").strip()

print("Environment file exists:", ENV_FILE.is_file(), flush=True)
print("Environment file loaded:", loaded, flush=True)
print("CARTO key available:", bool(CARTO_API_KEY), flush=True)

if ENABLE_CARTO_VOYAGER and not CARTO_API_KEY:
    raise RuntimeError(
        "CARTO key missing. Check your .env file or set "
        "ENABLE_CARTO_VOYAGER=False."
    )

ESRI_URL = (
    "https://server.arcgisonline.com/ArcGIS/rest/services/"
    "World_Imagery/MapServer/tile/{z}/{y}/{x}"
)

ESRI_TOPO_URL = (
    "https://server.arcgisonline.com/ArcGIS/rest/services/"
    "World_Topo_Map/MapServer/tile/{z}/{y}/{x}"
)

ESRI_ATTRIBUTION = (
    "Tiles &copy; Esri, Maxar, Earthstar Geographics, "
    "and GIS User Community"
)

CARTO_ATTRIBUTION = (
    '&copy; <a href="https://www.openstreetmap.org/copyright">'
    'OpenStreetMap</a> contributors, '
    '&copy; <a href="https://carto.com/attribution/">CARTO</a>'
)

def make_base_map(center, zoom=8, satellite_first=False, height=620):
    m = folium.Map(
        location=center,
        zoom_start=zoom,
        tiles=None,
        control_scale=True,
        height=height
    )

    folium.TileLayer(
        tiles=ESRI_TOPO_URL,
        name="Esri Topographic",
        attr="Tiles &copy; Esri and its data providers",
        max_zoom=19,
        show=(not satellite_first and not ENABLE_CARTO_VOYAGER),
        overlay=False,
        control=True
    ).add_to(m)

    if ENABLE_CARTO_VOYAGER:
        folium.TileLayer(
            tiles=(
                "https://basemaps.cartocdn.com/rastertiles/voyager/"
                "{z}/{x}/{y}.png?key=" + CARTO_API_KEY
            ),
            name="CARTO Voyager (OSM-derived)",
            attr=CARTO_ATTRIBUTION,
            max_zoom=20,
            show=not satellite_first,
            overlay=False,
            control=True
        ).add_to(m)

    folium.TileLayer(
        tiles=ESRI_URL,
        name="Esri Satellite",
        attr=ESRI_ATTRIBUTION,
        max_zoom=19,
        show=satellite_first,
        overlay=False,
        control=True
    ).add_to(m)

    return m


def ee_tile_layer(image, vis, name, shown=True, opacity=0.8):
    map_id = ee.Image(image).getMapId(vis)

    return folium.TileLayer(
        tiles=map_id["tile_fetcher"].url_format,
        attr="Google Earth Engine",
        name=name,
        show=shown,
        opacity=opacity,
        overlay=True,
        control=True
    )

print("Cell 7A completed: Basemaps configured.")

Environment file exists: True
Environment file loaded: True
CARTO key available: True
Cell 7A completed: Basemaps configured.


### MERIT Hydro and HydroBASINS layers

In [12]:
# Prepare Earth Engine datasets
river_major = MERIT.select("upa").gte(60).selfMask()
river_dense = MERIT.select("upa").gte(15).selfMask()

hyb_national = HYB.filterBounds(SRI_LANKA.geometry())

hyb_lines = hyb_national.style(
    color="b26acb",
    width=1,
    fillColor="00000000"
)

# Create national map
national = make_base_map(
    center=[7.65, 80.8],
    zoom=8,
    height=680
)

print("1. Loading MERIT dense river layer...", flush=True)

ee_tile_layer(
    river_dense,
    {"palette": ["65c5ed"]},
    "MERIT rivers — ≥15 km²",
    False,
    0.70
).add_to(national)

print("2. Loading MERIT major river layer...", flush=True)

ee_tile_layer(
    river_major,
    {"palette": ["16b7eb"]},
    "MERIT rivers — ≥60 km²",
    True,
    0.90
).add_to(national)

print("3. Loading HydroBASINS boundaries...", flush=True)

ee_tile_layer(
    hyb_lines,
    {},
    "HydroBASINS L12 boundaries",
    True,
    0.65
).add_to(national)

print("Cell 7B completed: All Earth Engine layers configured.", flush=True)

1. Loading MERIT dense river layer...
2. Loading MERIT major river layer...
3. Loading HydroBASINS boundaries...
Cell 7B completed: All Earth Engine layers configured.


### All 40 gauges and HTML map

In [13]:
import webbrowser

# All 40 ArcGIS dashboard gauges
all_gauges_layer = folium.FeatureGroup(
    name="All 40 ArcGIS dashboard gauges (19 listed basin groups)",
    show=True
)

for row in ALL_ARCGIS_GAUGES.itertuples(index=False):
    folium.CircleMarker(
        location=[float(row.lat), float(row.lon)],
        radius=4.7,
        color="#f9a602",
        fill=True,
        fill_color="#f9a602",
        fill_opacity=0.94,
        tooltip=f"{row.station} | {row.basin}",
        popup=(
            f"<b>{row.station}</b><br>"
            f"Dashboard basin: {row.basin}<br>"
            f"lat: {row.lat:.6f}, lon: {row.lon:.6f}"
        )
    ).add_to(all_gauges_layer)

all_gauges_layer.add_to(national)

print("Added", len(ALL_ARCGIS_GAUGES), "ArcGIS gauges.", flush=True)

# Optional 18 research-basin representatives
representatives = folium.FeatureGroup(
    name="18 research basin representatives (optional)",
    show=False
)

for row in RESEARCH_BASIN_MARKERS.itertuples(index=False):
    folium.CircleMarker(
        [row.lat, row.lon],
        radius=5,
        color="#45ad5e",
        fill=True,
        fill_opacity=0.9,
        tooltip=f"{row.basin} | {row.station}"
    ).add_to(representatives)

representatives.add_to(national)

# Optional seven Kelani source points
kelani_markers = folium.FeatureGroup(
    name="7 Kelani ArcGIS source points",
    show=False
)

for row in KELANI_GAUGES.itertuples(index=False):
    folium.CircleMarker(
        [row.lat, row.lon],
        radius=5,
        color="#e44b49",
        fill=True,
        tooltip=row.station
    ).add_to(kelani_markers)

kelani_markers.add_to(national)

# Interactive controls
plugins.Fullscreen().add_to(national)

plugins.MousePosition(
    position="bottomleft",
    num_digits=6,
    prefix="WGS84 cursor lat/lon:"
).add_to(national)

folium.LayerControl(collapsed=False).add_to(national)

# Save HTML map
national_map_html = (
    OUTPUT_DIR / "national_merit_hydrobasins_map.html"
).resolve()

national.save(str(national_map_html))

print("National map saved:", national_map_html, flush=True)
print("Displayed gauges:", len(ALL_ARCGIS_GAUGES), flush=True)

# Open in browser
opened = webbrowser.open_new_tab(national_map_html.as_uri())

if not opened:
    print("Open the HTML file manually in Chrome or Edge.")

print("Cell 7C completed.", flush=True)

Added 40 ArcGIS gauges.
National map saved: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\kelani_merit_hybas12_pilot\national_merit_hydrobasins_map.html
Displayed gauges: 40
Cell 7C completed.


## C. Save a numerical coordinate/network compatibility screen

For the 18 study-basin representatives, we record MERIT `upa` at the **original dashboard point** and a HydroBASINS Level 12 sub-basin containing that point, if available. This helps catch CRS/order failures and absent coverage. `UP_AREA` is **not expected to equal** MERIT `upa` at the exact gauge: Level 12 represents larger pre-defined sub-basins, not exact gauge outlets.

In [14]:
geod = Geod(ellps='WGS84')
checks=[]
for r in RESEARCH_BASIN_MARKERS.itertuples(index=False):
    pt=ee.Geometry.Point([float(r.lon),float(r.lat)])
    sample=MERIT.select(['dir','upa']).reduceRegion(
        reducer=ee.Reducer.first(),geometry=pt,scale=93,maxPixels=1000).getInfo()
    matches=HYB.filterBounds(pt)
    n=matches.size().getInfo()
    props=ee.Feature(matches.first()).toDictionary(['HYBAS_ID','SUB_AREA','UP_AREA','NEXT_DOWN']).getInfo() if n else {}
    checks.append({'basin':r.basin,'representative_station':r.station,'latitude':r.lat,'longitude':r.lon,
                   'merit_dir_nearest':sample.get('dir'), 'merit_upa_point_km2':sample.get('upa'),
                   'hybas12_matches':n,'hybas12_id':props.get('HYBAS_ID'),
                   'hybas12_up_area_km2_NOT_directly_comparable':props.get('UP_AREA')})
coverage=pd.DataFrame(checks)
coverage.to_csv(OUTPUT_DIR/'national_18basin_compatibility_screen.csv',index=False)
display(coverage)
assert len(coverage)==18
print('This is a CRS/coverage screen, not a hydrological proof of compatible catchment boundaries.')

,basin,representative_station,latitude,longitude,merit_dir_nearest,merit_upa_point_km2,hybas12_matches,hybas12_id,hybas12_up_area_km2_NOT_directly_comparable
0,Kelani Ganga,Nagalagam Street,6.958265,79.878642,64,2272.615967,1,4121601920,2258.8
1,Kalu Ganga,Putupaula,6.614324,80.061511,1,0.033968,1,4121159120,2753.4
2,Gin Ganga,Baddegama,6.173806,80.175381,64,0.008499,1,4121602610,821.9
3,Nilwala Ganga,Panadugama,6.111780,80.482384,16,434.147125,1,4121160080,603.5
4,Walawe Ganga,Moraketiya,6.345339,80.901244,2,0.008497,1,4121159660,1593.8
5,Kirindi Oya,Thanamalwila,6.469050,81.133751,4,697.067017,1,4121602330,835.4
6,Menik Ganga,Katharagama,6.417705,81.332190,32,0.110440,1,4121159500,940.9
7,Kumbukkan Oya,Nakkala,6.895362,81.297065,4,219.888168,1,4121602030,277.7
8,Heda Oya,Siyambalanduwa,6.905911,81.545410,4,289.428314,1,4121601990,312.6
9,Maduru Oya,Padiyathalawa,7.382251,81.194637,128,167.821426,1,4121601570,284.4


This is a CRS/coverage screen, not a hydrological proof of compatible catchment boundaries.


## D. Export native MERIT D8 and drainage-area raster for the Kelani system

We use a **regional processing envelope**, not a gauge-centred box. It must fully contain the actual upper catchments. The envelope is deliberately broader than the seven gauge points, **each derived watershed is later checked for touching any export edge**. If an edge is touched, the candidate is **truncated** and we must expand the envelope and rerun.

**Critical grid safeguard:** We obtain the source projection/affine transformation from MERIT, we download small, adjacent tiles **at that exact pixel grid** (no bilinear resampling, no user-defined 90 m reprojection). The source `dir` values must stay categorical `{0,1,2,4,8,16,32,64,128,-1}`. Downloading via Earth Engine is capped per request, so we tile the regional window and keep a resumable local GeoTIFF. Only GEE raster **data** needed for Kelani are downloaded, the national map uses GEE visualization tiles. <https://developers.google.com/earth-engine/apidocs/ee-image-getdownloadurl>

In [15]:
# WGS84 processing envelope covering the seven Kelani stations and expected headwaters.
# Inspect against the national map, do not infer that this rectangle itself is a watershed.
KELANI_EXPORT_BOUNDS = (79.52,6.46,81.08,7.56) # west, south, east, north
TILE_SIZE = 640       # small enough to remain below EE getDownloadURL size limit
RIVER_MIN_UPA_KM2 = 3 # candidate river cell screen, not the official stream definition
SEARCH_RADIUS_M = 350 # primary candidate search, a later fallback is FLAGGED for manual review
FALLBACK_RADIUS_M = 900
# Optional pixel-specific overrides after examining the candidate table/map.
# Example: {'Hanwella': (row_index, column_index)} using displayed global MERIT grid indices.
OUTLET_PIXEL_OVERRIDES = {}

proj = MERIT.select('dir').projection().getInfo()
src_crs = proj['crs']; src_affine = Affine(*proj['transform'])
if src_crs != 'EPSG:4326' or abs(src_affine.b)>1e-12 or abs(src_affine.d)>1e-12:
    raise RuntimeError(f'Unexpected MERIT native projection/rotation: {proj}. Stop and inspect before exporting.')
if not (src_affine.a > 0 and src_affine.e < 0):
    raise RuntimeError('Unexpected MERIT affine orientation; cannot align native-grid export.')
west,south,east,north=KELANI_EXPORT_BOUNDS
col0=int(math.floor((west-src_affine.c)/src_affine.a))
col1=int(math.ceil((east-src_affine.c)/src_affine.a))
row0=int(math.floor((north-src_affine.f)/src_affine.e))
row1=int(math.ceil((south-src_affine.f)/src_affine.e))
H=row1-row0;W=col1-col0
local_affine=src_affine*Affine.translation(col0,row0)
assert W>0 and H>0 and W<7000 and H<7000, (W,H)
print('Native-grid Kelani raster:',W,'columns x',H,'rows; estimated uncompressed MiB:',round(W*H*2*4/1024**2,1))
print('Exact source CRS:',src_crs,'native affine:',src_affine)
print('Exported window affine:',local_affine)

merged_path=OUTPUT_DIR/'merit_kelani_native_dir_upa.tif'
partial_path=OUTPUT_DIR/'merit_kelani_native_dir_upa_INCOMPLETE.tif'
if not merged_path.exists():
    merit_two = MERIT.select(['dir','upa']).toFloat().unmask(-9999)
    profile={'driver':'GTiff','height':H,'width':W,'count':2,'dtype':'float32',
             'crs':src_crs,'transform':local_affine,'nodata':-9999,
             'compress':'lzw','tiled':True,'blockxsize':256,'blockysize':256}
    with rasterio.open(partial_path,'w',**profile) as dest:
        dest.set_band_description(1,'MERIT dir ESRI D8 native values')
        dest.set_band_description(2,'MERIT upstream drainage area km2')
        for r0 in range(0,H,TILE_SIZE):
            for c0 in range(0,W,TILE_SIZE):
                th=min(TILE_SIZE,H-r0);tw=min(TILE_SIZE,W-c0)
                win=Window(c0,r0,tw,th)
                tf=window_transform(win,local_affine)
                parms={'crs':src_crs,'crs_transform':list(tuple(tf)[:6]),
                       'dimensions':[tw,th],'format':'GEO_TIFF'}
                tilepath=OUTPUT_DIR/f'_native_tile_r{r0}_c{c0}.tif'
                if not tilepath.exists():
                    # The URL is temporary. Persist the TIFF, never the URL.
                    url=merit_two.getDownloadURL(parms)
                    response=requests.get(url,timeout=180)
                    response.raise_for_status()
                    tmp_tile=tilepath.with_suffix('.partial')
                    tmp_tile.write_bytes(response.content)
                    with rasterio.open(tmp_tile) as check_tile:
                        if check_tile.count!=2 or check_tile.width!=tw or check_tile.height!=th:
                            raise RuntimeError('Incomplete GEE tile download: '+str(tmp_tile))
                    tmp_tile.replace(tilepath)
                with rasterio.open(tilepath) as tile:
                    if tile.count!=2 or tile.width!=tw or tile.height!=th:
                        raise RuntimeError('Unexpected MERIT tile dimensions/bands: '+str(tilepath))
                    if tile.crs.to_string()!=src_crs or not np.allclose(
                        tuple(tile.transform)[:6],tuple(tf)[:6],rtol=0,atol=1e-9):
                        raise RuntimeError('MERIT tile grid mismatch; STOP rather than corrupting D8 routing.')
                    dest.write(tile.read(),window=win)
                print(f'MERIT chunk r={r0}/{H} c={c0}/{W} verified')
    partial_path.replace(merged_path)
    print('Wrote verified complete raster:',merged_path)
else:
    print('Reusing native-grid MERIT raster:',merged_path)

with rasterio.open(merged_path) as src:
    assert src.count==2 and src.width==W and src.height==H
    assert tuple(src.transform)==tuple(local_affine)
    direction=src.read(1)
    upstream_area=src.read(2)
valid_dir={-1,0,1,2,4,8,16,32,64,128,-9999}
found=set(np.unique(direction[np.isfinite(direction)]).astype(int))
if not found.issubset(valid_dir):
    raise RuntimeError(f'Flow-pointer values altered by resampling: {found-valid_dir}')
if not np.allclose(direction,np.round(direction),atol=1e-5):
    raise RuntimeError('MERIT dir became fractional; do not perform watershed tracing.')
dir_grid=direction.astype(np.int16)
print('Native MERIT grid ready. Unique flow pointer values:',sorted(found))

Native-grid Kelani raster: 1873 columns x 1321 rows; estimated uncompressed MiB: 18.9
Exact source CRS: EPSG:4326 native affine: | 0.00, 0.00,-180.00|
| 0.00,-0.00, 85.00|
| 0.00, 0.00, 1.00|
Exported window affine: | 0.00, 0.00, 79.52|
| 0.00,-0.00, 7.56|
| 0.00, 0.00, 1.00|
MERIT chunk r=0/1321 c=0/1873 verified
MERIT chunk r=0/1321 c=640/1873 verified
MERIT chunk r=0/1321 c=1280/1873 verified
MERIT chunk r=640/1321 c=0/1873 verified
MERIT chunk r=640/1321 c=640/1873 verified
MERIT chunk r=640/1321 c=1280/1873 verified
MERIT chunk r=1280/1321 c=0/1873 verified
MERIT chunk r=1280/1321 c=640/1873 verified
MERIT chunk r=1280/1321 c=1280/1873 verified
Wrote verified complete raster: c:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\kelani_merit_hybas12_pilot\merit_kelani_native_dir_upa.tif
Native MERIT grid ready. Unique flow pointer values: [np.int64(-9999), np.int64(0), np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16), np.int64(32), np.int64(64), np.int64(

## E. Select computational outlets near the seven immutable ArcGIS gauge coordinates

We search native MERIT cells around each original point. A cell is a **candidate** if it has a legal flow code, an upstream area above the stream screening threshold, and lies within the search radius. We rank candidates by proximity *and* similarity to the published **drainage-area** reference. This reference is used only for checking outlets, the **annual-report coordinates are not used**.

Automatic ranking cannot determine the correct branch at a confluence. The notebook prints each station's candidate cells, retains both positions and adds a second map for manual examination. Override the chosen cell in `OUTLET_PIXEL_OVERRIDES` when needed, do not silently move the source gauge coordinates.

In [17]:
def candidate_cells(station, radius_m=SEARCH_RADIUS_M):
    rr,cc=rowcol(local_affine,float(station.lon),float(station.lat))
    rr=int(rr);cc=int(cc)
    if not (0<=rr<H and 0<=cc<W):
        raise ValueError(f'Station {station.station} outside MERIT export envelope')
    dy=max(4,int(math.ceil(radius_m/90))+2)
    results=[]
    for r in range(max(0,rr-dy),min(H,rr+dy+1)):
        for c in range(max(0,cc-dy),min(W,cc+dy+1)):
            val=int(dir_grid[r,c]); area=float(upstream_area[r,c])
            if val not in (0,1,2,4,8,16,32,64,128) or not np.isfinite(area) or area<RIVER_MIN_UPA_KM2:
                continue
            lon,lat=xy(local_affine,r,c,offset='center')
            distance=geod.inv(float(station.lon),float(station.lat),lon,lat)[2]
            if distance>radius_m:continue
            ref=float(station.official_area_km2)
            # A ranking only; not proof of outlet correctness.
            area_err=abs(math.log(max(area,0.001)/ref))
            rank=area_err*2+distance/450
            results.append({'station':station.station,'grid_row':r,'grid_col':c,
             'snapped_lon':lon,'snapped_lat':lat,'distance_m':round(distance,2),
             'merit_upa_km2':area,'official_area_km2':ref,
             'area_difference_pct':100*abs(area-ref)/ref,
             'direction':val,'rank_score':rank})
    return sorted(results,key=lambda v:v['rank_score'])

all_candidates=[]; outlet_rows=[]
for st in KELANI_GAUGES.itertuples(index=False):
    opts=candidate_cells(st)
    used_radius=SEARCH_RADIUS_M
    if not opts:
        opts=candidate_cells(st,FALLBACK_RADIUS_M)
        used_radius=FALLBACK_RADIUS_M
    for i,rec in enumerate(opts[:12]):
        all_candidates.append(dict(rec,ranking=i+1,used_radius_m=used_radius))
    if st.station in OUTLET_PIXEL_OVERRIDES:
        override=tuple(OUTLET_PIXEL_OVERRIDES[st.station])
        selected=next((r for r in opts if (r['grid_row'],r['grid_col'])==override),None)
        if selected is None:
            raise ValueError(f'Override {st.station} must be a candidate inside the reviewed radius: {override}')
        decision='MANUAL_CELL_OVERRIDE'
    elif opts:
        selected=opts[0]
        decision='AUTO_RANKED_CANDIDATE_NOT_VERIFIED'
    else:
        raise RuntimeError(f'No plausible MERIT channel candidate for {st.station}. Expand map/region and inspect.')
    outlet_rows.append(dict(selected,original_lat=st.lat,original_lon=st.lon,
                            tributary=st.tributary,selection_method=decision,
                            used_radius_m=used_radius))

candidates_df=pd.DataFrame(all_candidates)
outlets_df=pd.DataFrame(outlet_rows)
candidates_df.to_csv(OUTPUT_DIR/'outlet_candidate_rankings.csv',index=False)
outlets_df.to_csv(OUTPUT_DIR/'selected_computational_outlets_CANDIDATE.csv',index=False)
display(outlets_df[['station','distance_m','merit_upa_km2','official_area_km2',
                    'area_difference_pct','grid_row','grid_col','selection_method']].round(2))
print('We verify channel identity/tributary against the map before accepting any derived catchment.')

local_map=make_base_map([6.98,80.28],zoom=10,satellite_first=True,height=630)
ee_tile_layer(MERIT.select('upa').gte(3).selfMask(),{'palette':['36c5ea']},
              'MERIT streams ≥3 km²',True,0.82).add_to(local_map)
ee_tile_layer(hyb_lines,{},'HydroBASINS L12',False,0.60).add_to(local_map)
fg_original=folium.FeatureGroup(name='Original ArcGIS gauges (red)',show=True)
fg_outlet=folium.FeatureGroup(name='Computational outlets (yellow)',show=True)
for rec in outlets_df.itertuples(index=False):
    folium.CircleMarker([rec.original_lat,rec.original_lon],radius=6,
        color='red',fill=True,tooltip=rec.station+' — original').add_to(fg_original)
    folium.CircleMarker([rec.snapped_lat,rec.snapped_lon],radius=5,
        color='orange',fill=True,tooltip=f'{rec.station} — candidate MERIT outlet',
        popup=f'candidate only | MERIT drainage area {rec.merit_upa_km2:.1f} km²').add_to(fg_outlet)
    folium.PolyLine([[rec.original_lat,rec.original_lon],[rec.snapped_lat,rec.snapped_lon]],
        color='yellow',weight=2,tooltip=f'{rec.station}: {rec.distance_m:.0f} m').add_to(fg_outlet)
fg_original.add_to(local_map);fg_outlet.add_to(local_map)
plugins.Fullscreen().add_to(local_map)
folium.LayerControl(collapsed=False).add_to(local_map)
import webbrowser

local_map_html = (
    OUTPUT_DIR / "kelani_computational_outlets_map.html"
).resolve()

local_map.save(str(local_map_html))

print("Kelani computational outlets map saved:", local_map_html)

webbrowser.open_new_tab(local_map_html.as_uri())

,station,distance_m,merit_upa_km2,official_area_km2,area_difference_pct,grid_row,grid_col,selection_method
0,Nagalagam Street,58.47,2272.62,2085.0,9.00,722,431,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
1,Hanwella,40.32,1824.24,1782.0,2.37,780,676,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
2,Glencourse,58.08,1522.12,1463.0,4.04,699,809,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
3,Kithulgala,45.37,426.56,383.0,11.37,682,1079,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
4,Holombuwa,39.86,153.65,155.0,0.87,446,896,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
5,Deraniyagala,12.21,180.02,183.0,1.63,761,983,AUTO_RANKED_CANDIDATE_NOT_VERIFIED
6,Norwood,43.48,95.24,97.0,1.82,864,1310,AUTO_RANKED_CANDIDATE_NOT_VERIFIED


We verify channel identity/tributary against the map before accepting any derived catchment.
Kelani computational outlets map saved: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\kelani_merit_hybas12_pilot\kelani_computational_outlets_map.html


True

## F. Trace seven upstream D8 watersheds — free Python, with explicit topology checks

We follow MERIT's ESRI D8 rule **in reverse**: for an outlet, every neighbouring cell whose flow pointer leads into it is added, repeatedly. This is the standard upstream watershed principle, implemented here without installing a platform-specific executable. The algorithm does **not** interpolate flow-direction cells, does not flood-fill arbitrary enclosed land, and never manufactures connections across nodata.

We compare (i) mask area from a geodesic polygon, (ii) MERIT `upa` at the computational outlet, and (iii) published station drainage area. **Touching the regional export boundary is a hard failure** because the source data might truncate the catchment. Failed candidates are marked as such rather than silently accepted.

In [18]:
# For each neighboring offset, the required outgoing ESRI D8 code to flow INTO the current cell.
UPSTREAM_NEIGHBORS = [
    (0,-1,1),(-1,-1,2),(-1,0,4),(-1,1,8),
    (0,1,16),(1,1,32),(1,0,64),(1,-1,128)
]

def trace_upstream_d8(flow, seed_r, seed_c):
    nrows,ncols=flow.shape
    if not(0<=seed_r<nrows and 0<=seed_c<ncols):
        raise ValueError('Outlet outside MERIT export')
    if int(flow[seed_r,seed_c]) not in (-1,0,1,2,4,8,16,32,64,128):
        raise ValueError('Invalid D8 value at outlet')
    mask=np.zeros(flow.shape,dtype=np.uint8)
    q=deque([(int(seed_r),int(seed_c))]);mask[seed_r,seed_c]=1
    while q:
        r,c=q.popleft()
        for dr,dc,pointer in UPSTREAM_NEIGHBORS:
            nr,nc=r+dr,c+dc
            if 0<=nr<nrows and 0<=nc<ncols and not mask[nr,nc] and int(flow[nr,nc])==pointer:
                mask[nr,nc]=1
                q.append((nr,nc))
    return mask

# Fixed synthetic unit check proves both east-flow and south-flow tracing.
_test=np.zeros((6,6),dtype=np.int16)
_test[1,1]=1; _test[1,2]=4; _test[2,2]=4;_test[3,2]=0
_synthetic=trace_upstream_d8(_test,3,2)
assert _synthetic.sum()==4 and all(_synthetic[r,c] for r,c in [(1,1),(1,2),(2,2),(3,2)])
print('PASS: D8 reverse-direction synthetic topology test')

# Derived outputs: generated as CANDIDATES until hydrological and visual QA are complete.
from shapely.validation import explain_validity
summary=[]
combined_features=[]
for st in outlets_df.itertuples(index=False):
    station=st.station
    slug=''.join(ch if ch.isalnum() else '_' for ch in station).strip('_').lower()
    watershed=trace_upstream_d8(dir_grid,int(st.grid_row),int(st.grid_col))
    touching=bool(np.any(watershed[0,:]) or np.any(watershed[-1,:]) or
                  np.any(watershed[:,0]) or np.any(watershed[:,-1]))
    inds=np.argwhere(watershed==1)
    if inds.size==0:raise RuntimeError('Empty watershed for '+station)
    minr,minc=inds.min(axis=0);maxr,maxc=inds.max(axis=0)
    inner=watershed[minr:maxr+1,minc:maxc+1]
    inner_tf=window_transform(Window(minc,minr,inner.shape[1],inner.shape[0]),local_affine)
    features=[shape(g) for g,v in shapes(inner, mask=(inner==1), transform=inner_tf) if int(v)==1]
    geom=unary_union(features)
    if geom.is_empty or not geom.is_valid:
        raise RuntimeError(f'Invalid catchment polygon for {station}: {explain_validity(geom)}')
    geodesic_area=abs(geod.geometry_area_perimeter(geom)[0])/1e6
    ref=float(st.official_area_km2)
    area_diff=100*abs(geodesic_area-ref)/ref
    merit_diff=100*abs(geodesic_area-st.merit_upa_km2)/max(st.merit_upa_km2,0.00001)
    if touching:
        status='INCOMPLETE_TOUCHES_EXPORT_EDGE'
    elif merit_diff>15:
        status='REVIEW_D8_VS_MERIT_UPA'
    elif area_diff>20:
        status='REVIEW_AREA_DIFFERENCE_GT20'
    elif st.distance_m>SEARCH_RADIUS_M:
        status='REVIEW_DISTANT_OUTLET'
    else:
        status='CANDIDATE_PENDING_VISUAL_QA'

    # Crop native-grid mask to the polygon's bounding raster extent for compact storage.
    tif_path=OUTPUT_DIR/f'{slug}_catchment_mask_CANDIDATE.tif'
    with rasterio.open(tif_path,'w',driver='GTiff',height=inner.shape[0],width=inner.shape[1],count=1,
                       dtype='uint8',crs=src_crs,transform=inner_tf,compress='lzw') as dst:
        dst.write(inner,1)
        dst.set_band_description(1,'1 = upstream of candidate computational outlet; 0 = outside')
    props={'station':station,'tributary':st.tributary,'status':status,
           'source':'MERIT Hydro v1.0.1 ESRI D8; ArcGIS GeoJSON gauge geometry',
           'source_lon':float(st.original_lon),'source_lat':float(st.original_lat),
           'outlet_lon':float(st.snapped_lon),'outlet_lat':float(st.snapped_lat),
           'outlet_distance_m':float(st.distance_m),
           'official_area_km2':ref,'derived_area_km2':round(geodesic_area,3),
           'area_difference_pct':round(area_diff,2),
           'merit_upa_at_outlet_km2':round(float(st.merit_upa_km2),3),
           'derived_vs_merit_upa_pct':round(merit_diff,2)}
    polygon_fc={'type':'FeatureCollection','features':[{'type':'Feature','properties':props,'geometry':mapping(geom)}]}
    poly_path=OUTPUT_DIR/f'{slug}_catchment_polygon_CANDIDATE.geojson'
    poly_path.write_text(json.dumps(polygon_fc,ensure_ascii=False),encoding='utf-8')
    bounds=[float(v) for v in geom.bounds] # minx,miny,maxx,maxy
    rect=box(*bounds)
    bbox_fc={'type':'FeatureCollection','features':[{'type':'Feature','properties':dict(props,bbox_wgs84=bounds),
                   'geometry':mapping(rect)}]}
    bbox_path=OUTPUT_DIR/f'{slug}_bounding_box_CANDIDATE.geojson'
    bbox_path.write_text(json.dumps(bbox_fc,ensure_ascii=False),encoding='utf-8')
    rec={**props,'mask_file':tif_path.name,'polygon_file':poly_path.name,
         'bbox_file':bbox_path.name,'touches_edge':touching,'mask_cells':int(watershed.sum())}
    summary.append(rec)
    combined_features.append({'type':'Feature','properties':props,'geometry':mapping(geom)})
    print(f'{station}: {geodesic_area:.1f} km² (official {ref:.1f}; MERIT upa {st.merit_upa_km2:.1f}) — {status}')

results=pd.DataFrame(summary)
results.to_csv(OUTPUT_DIR/'kelani_7_catchment_QA_registry.csv',index=False)
(OUTPUT_DIR/'kelani_7_catchments_CANDIDATE_combined.geojson').write_text(
    json.dumps({'type':'FeatureCollection','features':combined_features},ensure_ascii=False),encoding='utf-8')
assert len(results)==7 and all((OUTPUT_DIR / f).exists() for col in ('mask_file','polygon_file','bbox_file') for f in results[col])
display(results[['station','derived_area_km2','official_area_km2','area_difference_pct',
                 'derived_vs_merit_upa_pct','outlet_distance_m','touches_edge','status']].round(2))
print('GENERATED: 7 candidate polygons, 7 native-grid masks, 7 bounding rectangles.')
print('IMPORTANT: none are labeled VERIFIED by the script. Manually review confluences and suspect statuses.')

PASS: D8 reverse-direction synthetic topology test
Nagalagam Street: 2272.8 km² (official 2085.0; MERIT upa 2272.6) — CANDIDATE_PENDING_VISUAL_QA
Hanwella: 1824.4 km² (official 1782.0; MERIT upa 1824.2) — CANDIDATE_PENDING_VISUAL_QA
Glencourse: 1522.2 km² (official 1463.0; MERIT upa 1522.1) — CANDIDATE_PENDING_VISUAL_QA
Kithulgala: 426.6 km² (official 383.0; MERIT upa 426.6) — CANDIDATE_PENDING_VISUAL_QA
Holombuwa: 153.7 km² (official 155.0; MERIT upa 153.6) — CANDIDATE_PENDING_VISUAL_QA
Deraniyagala: 180.0 km² (official 183.0; MERIT upa 180.0) — CANDIDATE_PENDING_VISUAL_QA
Norwood: 95.2 km² (official 97.0; MERIT upa 95.2) — CANDIDATE_PENDING_VISUAL_QA


,station,derived_area_km2,official_area_km2,area_difference_pct,derived_vs_merit_upa_pct,outlet_distance_m,touches_edge,status
0,Nagalagam Street,2272.76,2085.0,9.01,0.01,58.47,False,CANDIDATE_PENDING_VISUAL_QA
1,Hanwella,1824.35,1782.0,2.38,0.01,40.32,False,CANDIDATE_PENDING_VISUAL_QA
2,Glencourse,1522.21,1463.0,4.05,0.01,58.08,False,CANDIDATE_PENDING_VISUAL_QA
3,Kithulgala,426.59,383.0,11.38,0.01,45.37,False,CANDIDATE_PENDING_VISUAL_QA
4,Holombuwa,153.66,155.0,0.87,0.01,39.86,False,CANDIDATE_PENDING_VISUAL_QA
5,Deraniyagala,180.03,183.0,1.62,0.01,12.21,False,CANDIDATE_PENDING_VISUAL_QA
6,Norwood,95.24,97.0,1.81,0.01,43.48,False,CANDIDATE_PENDING_VISUAL_QA


GENERATED: 7 candidate polygons, 7 native-grid masks, 7 bounding rectangles.
IMPORTANT: none are labeled VERIFIED by the script. Manually review confluences and suspect statuses.


## G. Kelani visual QA map — inspect candidate polygons, basins, river cells and outlet placement

We display a separate local interactive map with individual candidate polygons, original gauge points and computational outlets. Toggle the basin and MERIT layers to inspect confluences, tributary membership and downstream exclusions. **These polygons are generated candidate watersheds, not independently certified catchments.** The map uses the same OSM and Esri Satellite basemaps.

In [19]:
qa_map=make_base_map([6.99,80.26],zoom=10,satellite_first=True,height=660)
ee_tile_layer(MERIT.select('upa').gte(5).selfMask(),{'palette':['48c9ee']},
              'MERIT river network',True,0.84).add_to(qa_map)
ee_tile_layer(hyb_lines,{},'HydroBASINS L12 boundaries',False,0.66).add_to(qa_map)
palette=['#fc8d59','#4dac26','#8073ac','#d7191c','#1b9e77','#e6ab02','#3182bd']
for index,rec in enumerate(results.itertuples(index=False)):
    fc=json.loads((OUTPUT_DIR/rec.polygon_file).read_text(encoding='utf-8'))
    geom=shape(fc['features'][0]['geometry'])
    # Simplify ONLY the on-screen line for browser performance; full-resolution geometry is saved.
    small=geom.simplify(0.00045,preserve_topology=True)
    layer=folium.FeatureGroup(name=f'{rec.station} candidate | {rec.status}',show=index<3)
    folium.GeoJson(mapping(small),style_function=lambda f,color=palette[index]:{
        'color':color,'weight':2.4,'fillColor':color,'fillOpacity':0.13},
        tooltip=f'{rec.station} — candidate upstream area {rec.derived_area_km2:.0f} km²').add_to(layer)
    folium.CircleMarker([rec.source_lat,rec.source_lon],color='red',radius=6,
        fill=True,tooltip=f'{rec.station} — original ArcGIS point').add_to(layer)
    folium.CircleMarker([rec.outlet_lat,rec.outlet_lon],color='yellow',radius=4,
        fill=True,tooltip=f'{rec.station} — candidate MERIT outlet').add_to(layer)
    layer.add_to(qa_map)
plugins.Fullscreen().add_to(qa_map)
folium.LayerControl(collapsed=False).add_to(qa_map)
# Save the local candidate-watershed QA map separately from notebook output.
import webbrowser
kelani_qa_html = (OUTPUT_DIR / 'kelani_catchment_qa_map.html').resolve()
qa_map.save(str(kelani_qa_html))
print('Kelani interactive QA map saved:', kelani_qa_html)
if not webbrowser.open_new_tab(kelani_qa_html.as_uri()):
    print('If the browser did not open, in VS Code right-click the HTML file')
    print('→ Reveal in File Explorer → open with Chrome or Edge.')

Kelani interactive QA map saved: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\kelani_merit_hybas12_pilot\kelani_catchment_qa_map.html


## H. Seven-gauge HydroBASINS cross-reference (not exact catchment equality)

We record the HydroBASINS Level 12 polygon containing each **selected computational outlet** and its watershed metadata. This checks that the same outlet falls inside a plausible mapped sub-basin. The value `UP_AREA` belongs to that HydroBASINS sub-basin and is **not** the exact MERIT gauge catchment drainage area; differences are expected. Ambiguous or zero matches are flagged rather than silently forced.

In [20]:
hyb_rows=[]
for st in outlets_df.itertuples(index=False):
    pt=ee.Geometry.Point([float(st.snapped_lon),float(st.snapped_lat)])
    subset=HYB.filterBounds(pt)
    count=int(subset.size().getInfo())
    props=(ee.Feature(subset.first()).toDictionary(['HYBAS_ID','SUB_AREA','UP_AREA','NEXT_DOWN']).getInfo()
           if count else {})
    hyb_rows.append({'station':st.station,'matched_hybas12_polygons':count,
        'hybas12_id':props.get('HYBAS_ID'),'hybas12_SUB_AREA_km2':props.get('SUB_AREA'),
        'hybas12_UP_AREA_km2_NOT_exact_gauge_area':props.get('UP_AREA'),
        'hybas12_NEXT_DOWN':props.get('NEXT_DOWN'),
        'status':'REVIEW_MULTIPLE_OR_NO_SUBBASIN' if count!=1 else 'INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED'})
hyb_qc=pd.DataFrame(hyb_rows)
hyb_qc.to_csv(OUTPUT_DIR/'kelani_7_hydrobasins_outlet_crosscheck.csv',index=False)
display(hyb_qc)

,station,matched_hybas12_polygons,hybas12_id,hybas12_SUB_AREA_km2,hybas12_UP_AREA_km2_NOT_exact_gauge_area,hybas12_NEXT_DOWN,status
0,Nagalagam Street,1,4121601920,140.4,2258.8,4120049050,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
1,Hanwella,1,4121158680,152.6,1818.8,4121601930,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
2,Glencourse,1,4121601910,170.8,1666.4,4121158680,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
3,Kithulgala,1,4121601880,144.7,544.1,4121158500,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
4,Holombuwa,1,4121601790,188.8,188.9,4121158490,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
5,Deraniyagala,1,4121601950,201.1,201.2,4121158550,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED
6,Norwood,1,4121158540,211.8,211.8,4121601880,INTERSECTION_FOUND_VISUAL_REVIEW_REQUIRED


## I. Publish the seven candidate catchments for the independent C1–C4 pipelines

We **retain the original processing outputs under `results/`** and copy only the seven watershed polygons, seven raster masks, seven bounding rectangles and QA metadata to `shared/spatial/catchments/kelani/`. This is an **idempotent local publishing step**.

The shared files retain the **`CANDIDATE`** suffix and original QA statuses: no catchment is automatically upgraded to VERIFIED. 

In [21]:
# LOCAL EXPORT ONLY — uses outputs already written by Sections F and H.
# Safe to rerun: same destination paths, matching files skipped; changed files replaced.
import csv
import hashlib
import shutil
from pathlib import Path
import pandas as pd
from zipfile import ZipFile, ZIP_DEFLATED

# Resolve paths locally even if the notebook kernel was restarted.
REPO_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
OUTPUT_DIR = REPO_ROOT / 'results' / 'catchments' / 'kelani_merit_hybas12_pilot'
SOURCE_DIR = OUTPUT_DIR
SHARE_DIR = REPO_ROOT / 'shared' / 'spatial' / 'catchments' / 'kelani'
ZIP_PATH = OUTPUT_DIR / 'kelani_7_catchments_CANDIDATE_team_share.zip'
EXPECTED_STATIONS = 7

qa_source = SOURCE_DIR / 'kelani_7_catchment_QA_registry.csv'
if not qa_source.is_file():
    raise FileNotFoundError(f'Run Section F once to create the QA registry: {qa_source}')
qa = pd.read_csv(qa_source)
if len(qa) != EXPECTED_STATIONS or qa['station'].nunique() != EXPECTED_STATIONS:
    raise RuntimeError(f'Expected 7 unique station records, found {len(qa)} rows.')
if qa[['polygon_file', 'mask_file', 'bbox_file']].isna().any().any():
    raise RuntimeError('Missing file names in QA registry; do not package.')
if qa['touches_edge'].astype(str).str.lower().eq('true').any():
    raise RuntimeError('At least one catchment touches MERIT export edge. Resolve before distributing.')

# Build the complete source->destination map FIRST. Abort before copying if incomplete.
planned = []
for rec in qa.itertuples(index=False):
    for column, subfolder in [
        ('polygon_file', 'polygons'),
        ('mask_file', 'masks'),
        ('bbox_file', 'bounding_boxes'),
    ]:
        filename = Path(getattr(rec, column)).name
        if '_CANDIDATE' not in filename:
            raise RuntimeError(f'Unexpected missing CANDIDATE marker: {filename}')
        planned.append((SOURCE_DIR / filename, SHARE_DIR / subfolder / filename))

for name in [
    'kelani_7_catchment_QA_registry.csv',
    'selected_computational_outlets_CANDIDATE.csv',
    'outlet_candidate_rankings.csv',
    'national_18basin_compatibility_screen.csv',
    'kelani_7_hydrobasins_outlet_crosscheck.csv',
    'kelani_7_catchments_CANDIDATE_combined.geojson',
]:
    if (SOURCE_DIR / name).exists():
        subfolder = 'metadata' if name != 'kelani_7_catchment_QA_registry.csv' else None
        dest = (SHARE_DIR / subfolder / name) if subfolder else SHARE_DIR / name
        planned.append((SOURCE_DIR / name, dest))

missing = [str(src) for src, _ in planned if not src.is_file()]
if missing:
    raise FileNotFoundError('Missing expected output file(s):\\n' + '\\n'.join(missing))
if len({str(dst).lower() for _, dst in planned}) != len(planned):
    raise RuntimeError('Duplicate destination paths detected; stopping without copying.')

def sha256_of(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

copied, reused = 0, 0
for source, target in planned:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and sha256_of(source) == sha256_of(target):
        reused += 1
        continue
    # Write to temporary file in same directory, then replace atomically.
    temporary = target.with_name(target.name + '.newcopy')
    shutil.copy2(source, temporary)
    temporary.replace(target)
    copied += 1

readme = SHARE_DIR / 'README.md'
readme.write_text('''# Kelani catchment candidates — FloodRisk Sri Lanka (J26-DS-314)

This folder contains **candidate**, not formally validated, MERIT Hydro v1.0.1
upstream catchments for seven Kelani gauge locations sourced from the public
Irrigation Department ArcGIS dashboard. HydroBASINS Level 12 is a geographic
cross-check rather than the primary delineation source.

- `polygons/`: irregular upstream watershed GeoJSON polygons (WGS84).
- `bounding_boxes/`: rectangular WGS84 bounding extents, **not** the watershed.
- `masks/`: native MERIT-grid GeoTIFF watershed masks, 1 inside and 0 outside.
  When resampling to another source grid, use a consistent categorical-mask
  procedure and preserve source raster NoData separately.
- `kelani_7_catchment_QA_registry.csv`: seven-station area/outlet/status audit.
- `metadata/`: outlet candidates, HydroBASINS and other supporting checks.
- `kelani_7_catchment_file_manifest.csv`: sizes and SHA256 hashes.

**Known limitations:** Kithulgala has about 11.38% difference from the published
drainage area; Nagalagam Street has about 9.01% and requires attention to
lowland/coastal river-network conditions. Internal traced-area agreement with
MERIT `upa` does NOT provide independent spatial validation. None of the seven
geometries is labelled VERIFIED.

**To share with the group:** Extract the team ZIP from `results/catchments/`
into the repository root. GeoTIFFs are intentionally ignored in Git, and must
be transferred separately. Do NOT upload the native regional MERIT raster,
intermediate tiles, or HTML maps with possible basemap/Earth Engine tokens.

**C1–C4:** All components may agree on these raw reference geometries and
station definitions. C4 independently builds and trains all three modality
pipelines, without C1/C2/C3 embeddings, features, models or predictions.
''', encoding='utf-8')

manifest_path = SHARE_DIR / 'kelani_7_catchment_file_manifest.csv'
manifest_records = []
for _, f in planned:
    manifest_records.append({
        'relative_path': f.relative_to(REPO_ROOT).as_posix(),
        'size_bytes': f.stat().st_size,
        'sha256': sha256_of(f),
        'source': 'MERIT Hydro v1.0.1 / ArcGIS gauges / HydroBASINS cross-check',
        'qa_status': 'CANDIDATE_NOT_VERIFIED',
    })
manifest_records.append({
    'relative_path': readme.relative_to(REPO_ROOT).as_posix(),
    'size_bytes': readme.stat().st_size,
    'sha256': sha256_of(readme),
    'source': 'Project methodology note',
    'qa_status': 'DOCUMENTATION',
})
with manifest_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['relative_path','size_bytes','sha256','source','qa_status'])
    writer.writeheader()
    writer.writerows(manifest_records)

# Overwrite one predictable ZIP path. No nested duplicate ZIP entries on rerun.
team_files = [target for _, target in planned] + [readme, manifest_path]
with ZipFile(ZIP_PATH, 'w', compression=ZIP_DEFLATED, compresslevel=6) as z:
    for f in sorted(team_files):
        z.write(f, arcname=f.relative_to(REPO_ROOT).as_posix())
with ZipFile(ZIP_PATH) as z:
    if z.testzip() is not None:
        raise RuntimeError('ZIP CRC validation failed.')
    names = z.namelist()
    if len(names) != len(set(names)):
        raise RuntimeError('Duplicate ZIP entries detected.')

print('Shared folder:', SHARE_DIR.resolve())
print(f'Stations: {len(qa)} | candidate polygons: 7 | masks: 7 | bounding boxes: 7')
print(f'Shared files newly copied/updated: {copied}; unchanged files reused: {reused}')
print('ZIP for the other three team members:', ZIP_PATH.resolve())
print('ZIP size (MiB):', round(ZIP_PATH.stat().st_size/1024**2, 2))
print('Manifest:', manifest_path)
print('IMPORTANT: Files remain CANDIDATE; final acceptance requires hydrological QA.')


Shared folder: C:\Users\ASUS\projects\rp_project\J26-DS-314\shared\spatial\catchments\kelani
Stations: 7 | candidate polygons: 7 | masks: 7 | bounding boxes: 7
Shared files newly copied/updated: 27; unchanged files reused: 0
ZIP for the other three team members: C:\Users\ASUS\projects\rp_project\J26-DS-314\results\catchments\kelani_merit_hybas12_pilot\kelani_7_catchments_CANDIDATE_team_share.zip
ZIP size (MiB): 0.13
Manifest: c:\Users\ASUS\projects\rp_project\J26-DS-314\shared\spatial\catchments\kelani\kelani_7_catchment_file_manifest.csv
IMPORTANT: Files remain CANDIDATE; final acceptance requires hydrological QA.
